<a href="https://colab.research.google.com/github/SyameimaruKoa/Minecraft-mod-AutoTranslation-tool/blob/main/run_colab_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minecraft Mod Auto Translator - Google Colab (LM Link)

このノートブックは、Google Colab 上で LM Studio (llmster) をバックグラウンドで起動し、**LM Link** を介してローカルPCの LM Studio と接続するためのものじゃ。

### 💡 LM Link の仕組みとメリット
- **ローカルPCで一元管理**: モデルのダウンロード、ロード、GPUオフロード設定などはすべて、そなたのローカルPCの LM Studio GUI からコントロールできるぞ。
- **設定不要**: Tailscale の認証キー登録や、面倒なポート開放設定などは一切不要じゃ。
- **シームレスな連携**: Colab とローカルPCをリンクすると、ローカルPCのモデル一覧にリモートモデルが表示される。ローカルの翻訳ツールは `localhost:1234`（デフォルト設定）に向けてリクエストを投げるだけで、自動的に Colab の GPU で推論が行われるのじゃ。

## 1. LM Studio (Headless版) のインストール

まずは Google Colab 上に LM Studio の CLI / デーモンをインストールするのじゃ。

In [ ]:
import os, subprocess, time
# 既存の古いデーモンプロセスを完全に終了（パスキー不一致による認証エラーを防ぐため）
!pkill -f lms || true
!pkill -f llmster || true
!pkill -f lmstudio || true
# Headless版 LM Studio のインストール
!curl -fsSL https://lmstudio.ai/install.sh | bash
# 環境変数PATHにlmsの実行フォルダを追加
os.environ["PATH"] = os.path.expanduser("~/.lmstudio/bin:") + os.environ["PATH"]
# パスなどのブートストラップを実行
!lms bootstrap
# ディスク書き込みを同期(Text file busyエラー防止)
!sync
# デーモン(llmster)をバックグラウンド起動
subprocess.Popen(["lms", "daemon", "up"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("[INFO] LM Studio Daemon (llmster) 起動完了じゃ。")

## 2. LM Link のログインと接続有効化

下のセルを実行すると、ログイン認証用のURLが表示されるぞ。
1. 表示されたURLをクリックし、ブラウザで LM Studio アカウントにログインして連携を承認するのじゃ。
2. 承認後、Colab上のLM Studioと、そなたのローカルPC上のLM Studioが自動的にリンクされるぞ。
3. リンク完了後は、ブラウザやこのColabタブは開いたままにしておくのじゃぞ（Colabインスタンスが起動し続けるため）。

In [ ]:
# アカウントログイン
!lms login
# LM Linkの有効化
!lms link enable

## 3. 翻訳モデルのダウンロードとロード

Google Colab (ホスト側) に翻訳用の LLM モデルをダウンロードし、GPU にロードして API サーバーを有効にするのじゃ。
これが完了すると、ローカルPCの LM Studio 上でこのモデルが「Linked (リンク済み)」として表示され、ローカルの翻訳ツールから使用可能になるぞ。

In [ ]:
# ダウンロードするモデル名 (GGUFリポジトリ名) を指定するのじゃ
model_identifier = "lmstudio-community/Qwen2.5-7B-Instruct-GGUF" # @param {type:"string"}
!lms get {model_identifier} < /dev/null
# GPUに最大レイヤーをオフロードしてロードするのじゃ
!lms load {model_identifier} --gpu=max < /dev/null
# APIサーバーを有効化するのじゃ
!lms server start < /dev/null